# Chapter 35: Vector Database Optimization & Log Analysis

> **What this notebook teaches**: How to search network logs by *meaning*, not just keywords — and how to detect anomalies automatically.

## The Semantic Gap Problem

Your network generates thousands of log lines per day. When an incident happens, you need to find all similar events. But the same authentication failure might appear as:

```
"Authentication failed for user admin"
"Login attempt rejected: invalid credentials"
"Access denied — bad password"
"SSH connection refused: auth error"
```

Four different strings. Same event. **Grep finds zero** when you search for any one.

**Vector databases** solve this by converting text meaning into numbers (vectors), so semantically similar logs land near each other in mathematical space — regardless of exact wording.

## 5 Demos You Will Build (Pure Python — No Special Libraries)

| Demo | Concept | What It Does |
|------|---------|-------------|
| 1 | **TF-IDF Vectors + Cosine Similarity** | Semantic search from scratch |
| 2 | **IVF Clustering (K-Means)** | Speed up search with cluster pre-filtering |
| 3 | **HNSW Graph Navigation** | Understand how fast ANN search works |
| 4 | **Hybrid Search (BM25 + Semantic + RRF)** | Combine keyword + meaning for best results |
| 5 | **Log Anomaly Detection** | Find logs that don't fit any known pattern |

> **Network Analogy**: A vector database does for meaning what a routing table does for destinations. It routes by semantic proximity, not by exact address match.


In [ ]:
import math
import re
import os
import random
import json
from collections import Counter, defaultdict

# ── Real network log corpus ───────────────────────────────────────────────────
# 20 logs across 4 categories: BGP, Auth failures, OSPF, Interface
NETWORK_LOGS = [
    # BGP events
    {"id": 0,  "text": "BGP peer 10.0.0.2 down: connection timeout after 90s",      "device": "core-rtr-01", "severity": "ERROR",   "category": "BGP"},
    {"id": 1,  "text": "BGP neighbor session reset — hold timer expired",             "device": "core-rtr-02", "severity": "ERROR",   "category": "BGP"},
    {"id": 2,  "text": "BGP adjacency with AS 65002 lost — TCP connection closed",   "device": "edge-rtr-01", "severity": "ERROR",   "category": "BGP"},
    {"id": 3,  "text": "BGP route withdrawal received from 10.0.0.2, 142 prefixes",  "device": "core-rtr-01", "severity": "WARNING", "category": "BGP"},
    {"id": 4,  "text": "Neighbor 10.0.0.3 changed BGP state from Established to Idle","device": "edge-rtr-02","severity": "ERROR",  "category": "BGP"},

    # Authentication failures
    {"id": 5,  "text": "Authentication failed for user admin from 192.168.1.50",      "device": "edge-fw-01",  "severity": "WARNING", "category": "AUTH"},
    {"id": 6,  "text": "Login rejected: invalid credentials, source 192.168.1.50",    "device": "mgmt-srv-01", "severity": "WARNING", "category": "AUTH"},
    {"id": 7,  "text": "SSH connection refused: authentication error from 192.168.1.50","device":"core-rtr-01","severity": "WARNING","category": "AUTH"},
    {"id": 8,  "text": "Access denied — bad password for user operator",              "device": "access-sw-01","severity": "WARNING", "category": "AUTH"},
    {"id": 9,  "text": "Failed login attempt: user root, 50 attempts in 5 minutes",  "device": "edge-fw-01",  "severity": "CRITICAL","category": "AUTH"},

    # OSPF events
    {"id": 10, "text": "OSPF neighbor 10.2.2.2 state changed from FULL to DOWN",     "device": "dist-sw-01",  "severity": "ERROR",   "category": "OSPF"},
    {"id": 11, "text": "OSPF adjacency lost on interface GigabitEthernet0/0",         "device": "core-rtr-01", "severity": "ERROR",   "category": "OSPF"},
    {"id": 12, "text": "OSPF dead interval expired — neighbor 10.2.2.3 unreachable", "device": "dist-sw-02",  "severity": "ERROR",   "category": "OSPF"},
    {"id": 13, "text": "OSPF Hello timer mismatch detected with neighbor 10.2.2.4",  "device": "access-sw-02","severity": "WARNING", "category": "OSPF"},
    {"id": 14, "text": "OSPF area 0 has lost a full-mesh adjacency with 3 routers",  "device": "core-rtr-02", "severity": "CRITICAL","category": "OSPF"},

    # Interface events
    {"id": 15, "text": "Interface GigabitEthernet0/1 changed state to down",          "device": "access-sw-03","severity": "ERROR",   "category": "LINK"},
    {"id": 16, "text": "Link down on port Ethernet1/5 — carrier lost",                "device": "dist-sw-01",  "severity": "ERROR",   "category": "LINK"},
    {"id": 17, "text": "Physical interface down: cable unplugged on Gi0/2",            "device": "access-sw-01","severity": "ERROR",   "category": "LINK"},
    {"id": 18, "text": "Port flapping detected on GigabitEthernet0/3 — 12 up/down in 1 min","device":"core-rtr-01","severity":"WARNING","category":"LINK"},
    {"id": 19, "text": "Interface bandwidth utilization at 51% — within normal range", "device": "edge-rtr-01", "severity": "INFO",    "category": "INFO"},
]

print(f"✓ Loaded {len(NETWORK_LOGS)} network logs")
print(f"  Categories: BGP={sum(1 for l in NETWORK_LOGS if l['category']=='BGP')}, "
      f"AUTH={sum(1 for l in NETWORK_LOGS if l['category']=='AUTH')}, "
      f"OSPF={sum(1 for l in NETWORK_LOGS if l['category']=='OSPF')}, "
      f"LINK={sum(1 for l in NETWORK_LOGS if l['category']=='LINK')}")
print()
print("Sample logs:")
for log in NETWORK_LOGS[:3]:
    print(f"  [{log['category']}] {log['text']}")


## Demo 1: TF-IDF Vectors + Cosine Similarity from Scratch

Before using any library, build a mini vector database by hand. This shows exactly what's happening inside tools like ChromaDB or FAISS.

### Two Key Concepts

**TF-IDF (Term Frequency — Inverse Document Frequency)**
Converts text to numbers. Important words (appear in few logs) get high weight. Common words (appear everywhere) get low weight.

```
"BGP" appears in 5/20 logs  → moderately important
"the" appears in 18/20 logs → not important (low weight)
"authentication" appears in 3/20 logs → very important
```

**Cosine Similarity**
Measures how similar two vectors are by the angle between them (not distance):
```
cos(θ) = (A · B) / (|A| × |B|)

Result:  1.0 = identical meaning
         0.5 = somewhat related
         0.0 = completely unrelated
        -1.0 = opposite meaning
```

> **Why cosine, not Euclidean distance?** A short log and a long log about the same event have similar direction (meaning) but different magnitude (length). Cosine ignores magnitude — direction only.


In [ ]:
# ── Demo 1: TF-IDF Vectors + Cosine Similarity ────────────────────────────────

def tokenize(text: str) -> list:
    """Simple tokenizer: lowercase, remove punctuation, split on whitespace."""
    text = text.lower()
    text = re.sub(r'[^a-z0-9./ ]', ' ', text)
    return [t for t in text.split() if len(t) > 1]

def build_tfidf_index(docs: list) -> tuple:
    """
    Build TF-IDF vectors for a list of text documents.
    Returns (vocab, doc_vectors) where each vector is a dict {word: tfidf_score}.
    """
    N = len(docs)

    # Step 1: Count how many documents each word appears in (document frequency)
    df = defaultdict(int)
    token_lists = []
    for doc in docs:
        tokens = set(tokenize(doc))   # set → count each word once per document
        token_lists.append(tokenize(doc))
        for token in tokens:
            df[token] += 1

    # Step 2: Build TF-IDF vector for each document
    vectors = []
    for tokens in token_lists:
        tf = Counter(tokens)
        total = sum(tf.values())
        vec = {}
        for word, count in tf.items():
            term_freq = count / total                       # TF: normalized
            inv_doc_freq = math.log(N / (df[word] + 1))    # IDF: rarer = higher
            vec[word] = term_freq * inv_doc_freq
        vectors.append(vec)

    return df, vectors

def cosine_similarity(vec_a: dict, vec_b: dict) -> float:
    """Compute cosine similarity between two sparse TF-IDF vectors."""
    # Dot product: only shared words contribute
    common_words = set(vec_a.keys()) & set(vec_b.keys())
    dot_product  = sum(vec_a[w] * vec_b[w] for w in common_words)

    # Magnitudes
    mag_a = math.sqrt(sum(v*v for v in vec_a.values()))
    mag_b = math.sqrt(sum(v*v for v in vec_b.values()))

    if mag_a == 0 or mag_b == 0:
        return 0.0
    return dot_product / (mag_a * mag_b)

class MiniVectorDB:
    """
    Minimal vector database using TF-IDF + cosine similarity.
    In production, replace vectorize() with a real embedding model
    (sentence-transformers, OpenAI embeddings, etc.)
    """
    def __init__(self):
        self.logs     = []
        self.vectors  = []

    def add_logs(self, log_list: list):
        texts = [l["text"] for l in log_list]
        _, vectors = build_tfidf_index(texts)
        self.logs    = log_list
        self.vectors = vectors
        print(f"  Indexed {len(log_list)} logs into vector DB")

    def search(self, query: str, top_k: int = 5, min_score: float = 0.05) -> list:
        """Find logs most semantically similar to the query."""
        _, [query_vec] = build_tfidf_index([query])
        scores = []
        for i, vec in enumerate(self.vectors):
            sim = cosine_similarity(query_vec, vec)
            if sim >= min_score:
                scores.append((sim, i))
        scores.sort(reverse=True)
        results = []
        for score, idx in scores[:top_k]:
            results.append({
                "score":    round(score, 4),
                "log":      self.logs[idx],
            })
        return results

# ── Build the mini vector DB ───────────────────────────────────────────────────
print("MINI VECTOR DB — TF-IDF + Cosine Similarity")
print("=" * 60)
db = MiniVectorDB()
db.add_logs(NETWORK_LOGS)

# ── Semantic searches ──────────────────────────────────────────────────────────
queries = [
    ("BGP peer unreachable",     "Should find BGP session events"),
    ("failed login brute force", "Should find AUTH events"),
    ("routing protocol down",    "Should find BGP and OSPF events"),
    ("cable disconnected",       "Should find interface/link events"),
]

for query, note in queries:
    print(f"\nQuery: '{query}'  ({note})")
    print(f"{'─'*55}")
    results = db.search(query, top_k=3)
    if not results:
        print("  No results above threshold")
    for r in results:
        log = r['log']
        print(f"  [{r['score']:.3f}] [{log['category']}] {log['text'][:65]}")

print()
print("KEY INSIGHT:")
print("  'BGP peer unreachable' found BGP events even though 'unreachable'")
print("  is not in the log text — shared BGP/network vocabulary creates similarity")
print("  In production: replace TF-IDF with sentence-transformers embeddings")
print("  for much better semantic matching across paraphrases.")


## Demo 2: IVF Clustering — Speed Up Search with K-Means

**The Problem**: Brute-force search compares every query against every stored vector. With 10 million logs at 768 dimensions each, that's billions of multiplications per query.

**The Solution**: Inverted File Index (IVF) — cluster logs into K groups first. At query time, only search the nearest 1-2 clusters (not all K).

### How IVF Works
```
INDEXING:
  1. Run K-Means on all log vectors → K cluster centroids
  2. Each log assigned to its nearest centroid
  3. Build an "inverted list" per centroid (which logs belong here)

QUERYING:
  1. Find which centroid is closest to query vector
  2. Search ONLY within that cluster's inverted list
  3. Return top matches from that cluster
```

### The Speed/Accuracy Tradeoff (nprobe)
- `nprobe=1` → search only closest cluster → fastest, misses edge cases
- `nprobe=3` → search 3 closest clusters → more accurate, 3× slower
- `nprobe=K` → search all clusters → exact result, same as brute force

> **Network Analogy**: IVF is like OSPF areas. Instead of flooding every router with every LSA, OSPF groups routers into areas — each router only needs to know details about its own area. Same principle: reduce the search space by pre-organizing into clusters.


In [ ]:
# ── Demo 2: IVF Clustering (K-Means from Scratch) ────────────────────────────

def vec_to_list(vec: dict, vocab: list) -> list:
    """Convert sparse dict vector to dense list using a fixed vocabulary."""
    return [vec.get(w, 0.0) for w in vocab]

def euclidean_dist(a: list, b: list) -> float:
    """Euclidean distance between two dense vectors."""
    return math.sqrt(sum((x-y)**2 for x, y in zip(a, b)))

def centroid_of(vectors: list) -> list:
    """Compute mean centroid of a list of dense vectors."""
    if not vectors:
        return []
    dim = len(vectors[0])
    return [sum(v[i] for v in vectors) / len(vectors) for i in range(dim)]

def kmeans(dense_vecs: list, k: int, max_iters: int = 10) -> tuple:
    """Simple K-Means clustering. Returns (centroids, assignments)."""
    # Initialize: pick K random vectors as starting centroids
    random.seed(42)
    centroids = random.sample(dense_vecs, k)

    assignments = [0] * len(dense_vecs)
    for iteration in range(max_iters):
        # Assign each vector to nearest centroid
        new_assignments = []
        for vec in dense_vecs:
            dists = [euclidean_dist(vec, c) for c in centroids]
            new_assignments.append(dists.index(min(dists)))

        # Check convergence
        if new_assignments == assignments:
            break
        assignments = new_assignments

        # Update centroids
        for ci in range(k):
            cluster_vecs = [dense_vecs[i] for i, a in enumerate(assignments) if a == ci]
            if cluster_vecs:
                centroids[ci] = centroid_of(cluster_vecs)

    return centroids, assignments

class IVFIndex:
    """
    Inverted File Index using K-Means pre-clustering.
    Demonstrates the speed/accuracy tradeoff with nprobe.
    """
    def __init__(self, k: int = 4):
        self.k         = k
        self.centroids = []
        self.clusters  = defaultdict(list)   # cluster_id → [(vec, log), ...]
        self.vocab     = []

    def build(self, logs: list, vectors: list):
        # Build vocabulary from all vectors
        all_words = set()
        for v in vectors:
            all_words.update(v.keys())
        self.vocab = sorted(all_words)

        # Convert to dense vectors
        dense = [vec_to_list(v, self.vocab) for v in vectors]

        # Run K-Means
        self.centroids, assignments = kmeans(dense, self.k)

        # Build inverted lists
        for i, (assignment, log) in enumerate(zip(assignments, logs)):
            self.clusters[assignment].append((dense[i], log))

        # Report cluster sizes
        for ci in range(self.k):
            cluster_logs = self.clusters[ci]
            cats = Counter(l['category'] for _, l in cluster_logs)
            print(f"  Cluster {ci}: {len(cluster_logs)} logs — {dict(cats)}")

    def search(self, query_vec: dict, top_k: int = 3, nprobe: int = 1) -> list:
        """
        Search only in the nprobe closest clusters.
        nprobe=1 → fast, nprobe=k → exact (brute force).
        """
        query_dense = vec_to_list(query_vec, self.vocab)

        # Find nprobe nearest centroids
        centroid_dists = [(euclidean_dist(query_dense, c), ci)
                          for ci, c in enumerate(self.centroids)]
        centroid_dists.sort()
        search_clusters = [ci for _, ci in centroid_dists[:nprobe]]

        # Search within selected clusters using cosine similarity
        candidates = []
        for ci in search_clusters:
            for vec, log in self.clusters[ci]:
                sim = cosine_similarity(
                    dict(zip(self.vocab, vec)),
                    dict(zip(self.vocab, query_dense))
                )
                candidates.append((sim, log))

        candidates.sort(reverse=True)
        return [{"score": round(s, 4), "log": l} for s, l in candidates[:top_k]]

# ── Build and test IVF index ───────────────────────────────────────────────────
print("IVF CLUSTERING — K-Means with nprobe Search")
print("=" * 60)

texts = [l["text"] for l in NETWORK_LOGS]
_, tfidf_vecs = build_tfidf_index(texts)

ivf = IVFIndex(k=4)
print(f"Building IVF index with K={ivf.k} clusters...")
ivf.build(NETWORK_LOGS, tfidf_vecs)

# Build query vector
_, [query_vec] = build_tfidf_index(["BGP neighbor connection lost"])

print()
print("NPROBE COMPARISON (same query, different search scope):")
print(f"Query: 'BGP neighbor connection lost'")
print()

for nprobe in [1, 2, 4]:
    results = ivf.search(query_vec, top_k=3, nprobe=nprobe)
    categories = [r['log']['category'] for r in results]
    print(f"nprobe={nprobe}: {len(results)} results — categories: {categories}")
    for r in results:
        print(f"  [{r['score']:.3f}] {r['log']['text'][:60]}")

print()
print("TRADEOFF:")
print("  nprobe=1 → fastest, may miss logs at cluster boundaries")
print("  nprobe=4 → same as brute force, perfect recall, 4x slower")
print("  nprobe=2 → good balance for most network log workloads")


## Demo 3: HNSW — How Graph-Based ANN Search Works

**HNSW (Hierarchical Navigable Small World)** is the algorithm behind most modern vector databases (Pinecone, Weaviate, Qdrant). It builds a multi-layer graph where:

- **Top layer**: few nodes, long edges → fast coarse navigation
- **Bottom layer**: all nodes, short edges → precise fine-grained search

### Navigation Process
```
Query arrives
    │
[Layer 2 - sparse]  Entry point → jump to nearest node (fast, big steps)
    │
[Layer 1 - medium]  Continue navigating → getting closer
    │
[Layer 0 - dense]   Fine-grained search → find actual nearest neighbors
    │
Return Top-K results
```

### Key Parameters
| Parameter | Effect |
|-----------|--------|
| `M` (connections per node) | Higher M = better accuracy, more memory |
| `ef_search` | Larger candidate list = better recall, slower |
| `ef_construction` | Build quality — set high during indexing |

> **The "Small World" Property**: From any node in the graph, you can reach any other node in a small number of hops. Like social networks — you can reach any person in ~6 steps. HNSW exploits this for fast navigation.


In [ ]:
# ── Demo 3: Simplified HNSW Graph Navigation Demo ────────────────────────────
# This shows the CONCEPT of HNSW — not a full implementation (that would be 1000+ lines)
# Real HNSW is inside FAISS, Qdrant, Weaviate, etc.

class HNSWDemo:
    """
    Simplified 2-layer HNSW to demonstrate the navigation concept.
    Layer 1: sparse (M=2 connections per node — coarse navigation)
    Layer 0: dense (M=4 connections per node — fine search)
    """

    def __init__(self, M: int = 3, ef_search: int = 5):
        self.M         = M          # max connections per node
        self.ef_search = ef_search  # candidate list size during search
        self.layer0    = {}         # bottom layer: all nodes + connections
        self.layer1    = {}         # top layer: subset of nodes + long edges
        self.nodes     = []         # stored (vector, log) pairs
        self.entry_point = None     # starting node for search

    def _dist(self, a: dict, b: dict) -> float:
        """Cosine distance (1 - similarity) — smaller = more similar."""
        sim = cosine_similarity(a, b)
        return 1.0 - sim

    def add(self, vec: dict, log: dict):
        """Add a node to the graph. Connects to M nearest existing nodes."""
        idx = len(self.nodes)
        self.nodes.append((vec, log))

        if idx == 0:
            self.entry_point = 0
            self.layer0[idx] = []
            # ~30% of nodes go into layer1 (sparse layer)
            if random.random() < 0.3:
                self.layer1[idx] = []
            return

        # Find M nearest neighbors in layer0 to connect to
        dists = [(self._dist(vec, self.nodes[i][0]), i)
                 for i in range(idx)]
        dists.sort()
        neighbors = [i for _, i in dists[:self.M]]

        # Add bidirectional edges in layer0
        self.layer0[idx] = neighbors
        for n in neighbors:
            if idx not in self.layer0.get(n, []):
                self.layer0.setdefault(n, []).append(idx)

        # ~30% of nodes get into layer1 (sparse long-range connections)
        if random.random() < 0.3:
            layer1_neighbors = [i for _, i in dists[:2] if i in self.layer1]
            self.layer1[idx] = layer1_neighbors
            for n in layer1_neighbors:
                if idx not in self.layer1.get(n, []):
                    self.layer1.setdefault(n, []).append(idx)

    def search(self, query_vec: dict, top_k: int = 3) -> list:
        """Navigate from top layer to bottom, tracking hops for visualization."""
        if not self.nodes:
            return []

        visited = set()
        hops_log = []

        # ── Phase 1: Coarse navigation in Layer 1 ────────────────────────
        current = self.entry_point
        best_dist = self._dist(query_vec, self.nodes[current][0])

        if self.layer1:
            for _ in range(10):   # max iterations
                improved = False
                for neighbor in self.layer1.get(current, []):
                    if neighbor not in visited:
                        d = self._dist(query_vec, self.nodes[neighbor][0])
                        if d < best_dist:
                            best_dist = d
                            current = neighbor
                            improved = True
                visited.add(current)
                if not improved:
                    break
            hops_log.append(f"Layer1 coarse nav → node {current} (dist={best_dist:.3f})")

        # ── Phase 2: Fine-grained search in Layer 0 ──────────────────────
        # Beam search with ef_search candidate list
        candidates = [(best_dist, current)]
        visited_l0 = {current}
        results    = [(best_dist, current)]

        while candidates:
            dist, node = min(candidates)
            candidates.remove((dist, node))

            # Explore neighbors in layer0
            for neighbor in self.layer0.get(node, []):
                if neighbor not in visited_l0:
                    visited_l0.add(neighbor)
                    d = self._dist(query_vec, self.nodes[neighbor][0])
                    candidates.append((d, neighbor))
                    results.append((d, neighbor))
                    # Trim candidate list to ef_search size
                    if len(candidates) > self.ef_search:
                        candidates.remove(max(candidates))

        hops_log.append(f"Layer0 fine search explored {len(visited_l0)} nodes")

        # Sort and return top_k
        results.sort()
        output = []
        for dist, idx in results[:top_k]:
            output.append({
                "score": round(1 - dist, 4),
                "log":   self.nodes[idx][1],
                "hops":  hops_log
            })
        return output

# ── Build HNSW and run searches ───────────────────────────────────────────────
print("HNSW GRAPH NAVIGATION DEMO")
print("=" * 60)

random.seed(42)
texts = [l["text"] for l in NETWORK_LOGS]
_, tfidf_vecs = build_tfidf_index(texts)

hnsw = HNSWDemo(M=3, ef_search=8)
print(f"Building HNSW (M={hnsw.M}, ef_search={hnsw.ef_search})...")
for vec, log in zip(tfidf_vecs, NETWORK_LOGS):
    hnsw.add(vec, log)

layer0_size = len(hnsw.layer0)
layer1_size = len(hnsw.layer1)
print(f"  Layer0 (dense):  {layer0_size} nodes")
print(f"  Layer1 (sparse): {layer1_size} nodes (~{100*layer1_size//layer0_size}% of total)")
print()

# Test queries
test_queries = [
    "BGP session dropped unexpectedly",
    "unauthorized access attempt",
]

_, query_vecs = build_tfidf_index(test_queries)
for query, qvec in zip(test_queries, query_vecs):
    print(f"Query: '{query}'")
    results = hnsw.search(qvec, top_k=3)
    print(f"  Navigation: {results[0]['hops'][0]}")
    print(f"  Navigation: {results[0]['hops'][1]}")
    for r in results:
        log = r['log']
        print(f"  [{r['score']:.3f}] [{log['category']}] {log['text'][:60]}")
    print()

print("HNSW PARAMETERS IN PRODUCTION:")
print("  M=16        → good default (increase for higher recall)")
print("  ef_search=64 → good default (increase for higher accuracy)")
print("  ef_construction=200 → set high during build, doesn't affect query speed")
print()
print("  Real HNSW lives inside: FAISS, Qdrant, Weaviate, Pinecone")
print("  Use them in production — this demo shows the concept only.")


## Demo 4: Hybrid Search — BM25 + Semantic + RRF

**Pure semantic search** struggles with exact identifiers — IP addresses, CVE numbers, interface names. "Find all events from 192.168.1.50" — semantic similarity might return events from 192.168.1.51 that are more "semantically similar" (same failure type, different IP).

**Pure keyword search (BM25)** misses paraphrases — "BGP peer down" won't find "BGP neighbor unreachable".

**Hybrid search** combines both, merged with **Reciprocal Rank Fusion (RRF)**:

### RRF Formula
```python
rrf_score = 1/(rank_semantic + 60) + 1/(rank_bm25 + 60)
```
- Each result gets a score from its rank in EACH list
- The constant 60 prevents top-rank results from dominating too much
- Results ranked highest in BOTH lists float to the top

### When Hybrid Beats Both
| Query Type | Best Search |
|-----------|------------|
| "BGP events similar to this" | Semantic wins |
| "events from IP 192.168.1.50" | BM25 wins |
| "failed BGP from 10.0.0.2" | **Hybrid wins** (both contribute) |


In [ ]:
# ── Demo 4: Hybrid Search — BM25 + Semantic + RRF ────────────────────────────

class BM25:
    """
    BM25 keyword relevance scorer (the algorithm behind Elasticsearch).
    Better than raw TF-IDF: normalizes for document length + diminishing returns on TF.
    """
    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1   = k1    # term saturation (1.2-2.0)
        self.b    = b     # length normalization (0.75 = standard)
        self.docs = []
        self.df   = defaultdict(int)
        self.avgdl = 0

    def fit(self, docs: list):
        """Index documents for BM25 scoring."""
        self.docs = [tokenize(d) for d in docs]
        self.avgdl = sum(len(d) for d in self.docs) / len(self.docs)
        for tokens in self.docs:
            for word in set(tokens):
                self.df[word] += 1

    def score(self, query: str, doc_idx: int) -> float:
        """BM25 relevance score for query against one document."""
        query_tokens = tokenize(query)
        doc_tokens   = self.docs[doc_idx]
        dl = len(doc_tokens)
        tf_map = Counter(doc_tokens)
        N = len(self.docs)
        score = 0.0
        for term in query_tokens:
            if term not in tf_map:
                continue
            tf  = tf_map[term]
            df  = self.df.get(term, 0)
            idf = math.log((N - df + 0.5) / (df + 0.5) + 1)
            tf_norm = (tf * (self.k1 + 1)) / (tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl))
            score += idf * tf_norm
        return score

    def search(self, query: str, top_k: int = 10) -> list:
        """Return ranked list of (score, doc_idx) pairs."""
        scores = [(self.score(query, i), i) for i in range(len(self.docs))]
        scores.sort(reverse=True)
        return [(s, i) for s, i in scores[:top_k] if s > 0]

def reciprocal_rank_fusion(semantic_results: list, bm25_results: list,
                            all_logs: list, top_k: int = 5, k: int = 60) -> list:
    """
    Merge semantic and BM25 ranked lists using RRF.
    RRF score = 1/(rank_semantic + k) + 1/(rank_bm25 + k)
    k=60 is the standard constant (reduces sensitivity to top ranks).
    """
    rrf_scores = defaultdict(float)

    # Semantic results: (score, log_idx) ordered best-first
    for rank, (score, idx) in enumerate(semantic_results):
        rrf_scores[idx] += 1.0 / (rank + k)

    # BM25 results: (score, log_idx) ordered best-first
    for rank, (score, idx) in enumerate(bm25_results):
        rrf_scores[idx] += 1.0 / (rank + k)

    # Sort by combined RRF score
    sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [(rrf_score, all_logs[idx]) for idx, rrf_score in sorted_results[:top_k]]

# ── Build both indexes ────────────────────────────────────────────────────────
print("HYBRID SEARCH — BM25 + Semantic + RRF Fusion")
print("=" * 60)

texts = [l["text"] for l in NETWORK_LOGS]
_, tfidf_vecs = build_tfidf_index(texts)

bm25 = BM25()
bm25.fit(texts)

# ── Test 3 query types to show when each approach wins ────────────────────────
test_cases = [
    {
        "query": "BGP neighbor connection dropped",
        "note":  "Semantic wins: paraphrases of BGP session issues"
    },
    {
        "query": "192.168.1.50",
        "note":  "BM25 wins: exact IP address match"
    },
    {
        "query": "failed authentication 192.168.1.50",
        "note":  "Hybrid wins: exact IP + semantic meaning combined"
    },
]

for tc in test_cases:
    query = tc["query"]
    print(f"\nQuery: '{query}'")
    print(f"Note:  {tc['note']}")
    print("─" * 55)

    # Semantic results
    _, [qvec] = build_tfidf_index([query])
    sem_scored = [(cosine_similarity(qvec, v), i)
                  for i, v in enumerate(tfidf_vecs)]
    sem_scored.sort(reverse=True)
    sem_top = [(s, i) for s, i in sem_scored[:10] if s > 0.01]

    # BM25 results
    bm25_top = bm25.search(query, top_k=10)

    # Hybrid RRF
    hybrid = reciprocal_rank_fusion(sem_top, bm25_top, NETWORK_LOGS, top_k=3)

    print(f"  Semantic top-2:  ", end="")
    print(", ".join([f"[{NETWORK_LOGS[i]['category']}]" for _, i in sem_top[:2]]))

    print(f"  BM25 top-2:      ", end="")
    print(", ".join([f"[{NETWORK_LOGS[i]['category']}]" for _, i in bm25_top[:2]]))

    print(f"  Hybrid top-3:")
    for rrf_score, log in hybrid:
        print(f"    [{rrf_score:.4f}] [{log['category']}] {log['text'][:60]}")

print()
print("HYBRID SEARCH RULE:")
print("  Always use hybrid when queries mix:")
print("  • Exact identifiers (IPs, interface names, CVE IDs) → BM25 handles these")
print("  • Semantic intent (similar events, same failure type) → Semantic handles these")


## Demo 5: Log Anomaly Detection with Semantic Clustering

**The Problem**: Signature-based alert rules only catch known patterns. Novel attack types, misconfiguration patterns, or failure modes that have never happened before won't match any rule.

**The Solution**: Cluster all normal logs in vector space. Any new log that lands **far from every cluster centroid** is an anomaly — semantically different from everything seen before.

### How It Works
```
TRAINING (on normal logs):
  Embed all historical logs → cluster into K groups
  Each cluster = a "normal log pattern"
  Store centroid of each cluster

DETECTION (on new log):
  Embed new log → find distance to nearest centroid
  If min_distance > threshold → ANOMALY 🚨
  If min_distance ≤ threshold → NORMAL ✓
```

### Threshold Tuning
- Too low → many false positives (normal logs flagged)
- Too high → misses real anomalies
- Strategy: set threshold at 95th percentile of training distances

> **Authentication Anomaly Use Case**: Embed all auth logs. Novel attack patterns (new tool, new source pattern) appear as outliers far from the clusters of known brute-force or credential-stuffing patterns.


In [ ]:
# ── Demo 5: Log Anomaly Detection ────────────────────────────────────────────

class VectorAnomalyDetector:
    """
    Detect anomalous logs using semantic distance from cluster centroids.
    Logs semantically far from ALL known patterns → anomaly.
    """

    def __init__(self, k_clusters: int = 4, anomaly_percentile: float = 95):
        self.k          = k_clusters
        self.percentile = anomaly_percentile
        self.centroids  = []
        self.vocab      = []
        self.threshold  = None
        self.training_distances = []

    def _vec_dist(self, a: dict, b_dense: list) -> float:
        """Distance from sparse dict vector to dense centroid vector."""
        a_dense = [a.get(w, 0.0) for w in self.vocab]
        return euclidean_dist(a_dense, b_dense)

    def _min_centroid_dist(self, vec: dict) -> float:
        """Distance from vector to its nearest cluster centroid."""
        dists = [self._vec_dist(vec, c) for c in self.centroids]
        return min(dists) if dists else float('inf')

    def fit(self, logs: list):
        """Train on normal logs: cluster them and set anomaly threshold."""
        texts = [l["text"] for l in logs]
        _, vecs = build_tfidf_index(texts)

        # Build vocabulary and convert to dense
        all_words = set()
        for v in vecs:
            all_words.update(v.keys())
        self.vocab = sorted(all_words)
        dense = [vec_to_list(v, self.vocab) for v in vecs]

        # K-Means clustering
        self.centroids, _ = kmeans(dense, self.k)

        # Compute distance of each training log to its nearest centroid
        self.training_distances = [
            self._min_centroid_dist(v) for v in vecs
        ]

        # Set threshold at the specified percentile of training distances
        sorted_dists = sorted(self.training_distances)
        p_idx = int(self.percentile / 100 * len(sorted_dists))
        self.threshold = sorted_dists[min(p_idx, len(sorted_dists)-1)]

        print(f"  Trained on {len(logs)} logs with K={self.k} clusters")
        print(f"  Anomaly threshold (p{self.percentile:.0f}): {self.threshold:.4f}")
        avg = sum(self.training_distances) / len(self.training_distances)
        print(f"  Avg training distance: {avg:.4f}")

    def detect(self, log_text: str) -> dict:
        """Score a new log. Returns anomaly result with distance and verdict."""
        _, [vec] = build_tfidf_index([log_text])
        dist = self._min_centroid_dist(vec)
        is_anomaly = dist > self.threshold
        anomaly_score = dist / self.threshold if self.threshold > 0 else 0
        return {
            "text":          log_text,
            "distance":      round(dist, 4),
            "threshold":     round(self.threshold, 4),
            "anomaly_score": round(anomaly_score, 2),
            "is_anomaly":    is_anomaly,
        }

# ── Train and test ─────────────────────────────────────────────────────────────
print("LOG ANOMALY DETECTION — Semantic Clustering")
print("=" * 60)

# Train on all normal logs
detector = VectorAnomalyDetector(k_clusters=4, anomaly_percentile=90)
print("Training on normal log corpus...")
detector.fit(NETWORK_LOGS)

# Test logs — mix of normal and anomalous
test_logs = [
    # Normal — similar to training data
    ("BGP neighbor 10.0.0.5 went to Idle state",           "normal"),
    ("SSH login failed for user cisco from 10.1.1.10",      "normal"),
    ("OSPF adjacency lost on Gi0/1 — hello timer expired",  "normal"),
    ("Interface Gi0/4 link down",                           "normal"),

    # Anomalous — semantically different from training data
    ("SNMP community string accepted from unknown OID .1.3.6.1.999.0", "anomaly"),
    ("BGP UPDATE message contains malformed path attribute type 255",    "anomaly"),
    ("Memory heap corruption detected in routing process",               "anomaly"),
    ("Unexpected reboot: watchdog timer expired — kernel panic",         "anomaly"),
    ("Configuration file modified via console without AAA logging",      "anomaly"),
]

print()
print(f"{'─'*65}")
print(f"{'Log (truncated)':<45} {'Score':>6} {'Verdict'}")
print(f"{'─'*65}")

normal_count   = 0
anomaly_count  = 0
correct        = 0

for log_text, true_label in test_logs:
    result = detector.detect(log_text)
    verdict = "🚨 ANOMALY" if result["is_anomaly"] else "✅ normal"
    score_bar = "█" * min(int(result["anomaly_score"] * 5), 10)
    print(f"  {log_text[:43]:<43} {result['anomaly_score']:>5.1f}x  {verdict}")

    if true_label == "anomaly":
        anomaly_count += 1
        if result["is_anomaly"]:
            correct += 1
    else:
        normal_count += 1
        if not result["is_anomaly"]:
            correct += 1

print(f"{'─'*65}")
print()
print(f"RESULTS:")
print(f"  Correct detections: {correct}/{len(test_logs)}")
print(f"  Threshold used:     {detector.threshold:.4f}")
print()
print("PRODUCTION USE:")
print("  1. Train on 30 days of normal logs (update weekly)")
print("  2. Set percentile based on acceptable false positive rate")
print("  3. Anomaly score > 2.0 → page on-call")
print("  4. Anomaly score 1.0-2.0 → log for analyst review")
print("  5. Feed anomaly log into LLM for explanation + remediation hint")


## Summary — Choosing Your Vector Search Strategy

| Use Case | Best Approach | Why |
|----------|--------------|-----|
| < 100K logs, any hardware | **Brute-force cosine** | Simple, exact, fast enough |
| 100K–10M logs | **IVF with nprobe=8** | Clusters reduce search space 100× |
| High recall required | **HNSW (M=16, ef=64)** | Best accuracy/speed tradeoff |
| Mix of semantic + exact queries | **Hybrid BM25 + Semantic + RRF** | Best of both worlds |
| Novel threat detection | **Cluster + distance threshold** | Catches what rules miss |

## Vector Database Tooling Reference

| Tool | Best For | Notes |
|------|---------|-------|
| **ChromaDB** | Prototyping, small scale | Easiest to set up, local |
| **FAISS** | High-performance, Python | Facebook's library, no cloud needed |
| **Qdrant** | Production, filtering | Excellent metadata filtering |
| **pgvector** | Already using PostgreSQL | SQL + vectors in one query |
| **Pinecone** | Fully managed cloud | Zero ops, pay per usage |

## The Production Checklist
- [ ] Normalize embeddings at ingestion (L2 normalize) → enables dot product instead of cosine
- [ ] Store metadata with every vector (device, severity, timestamp)
- [ ] Monitor Recall@K weekly against a ground-truth query set
- [ ] Separate ingestion workers from query workers (write/read isolation)
- [ ] Retrain anomaly detector weekly on fresh normal-log baseline


In [ ]:
# ── Full Integration: Smart Log Search Engine ─────────────────────────────────
# Combines: Vector DB + Hybrid Search + Anomaly Detection + LLM Analysis

import os
try:
    from anthropic import Anthropic
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    import getpass
    if not os.environ.get('ANTHROPIC_API_KEY'):
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')

    _API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
    client  = Anthropic() if _API_KEY else None
except ImportError:
    client = None

def llm_analyze(log_text: str, similar_logs: list) -> str:
    """Ask Claude to summarize what the anomalous log means + remediation."""
    if not client:
        return (
            "SIMULATED ANALYSIS:\n"
            f"  Log: {log_text[:60]}\n"
            "  This log is semantically distant from all known patterns.\n"
            "  Possible cause: Novel configuration issue or new attack vector.\n"
            "  Recommended action: Investigate immediately — escalate to senior engineer."
        )
    context = "\n".join([f"- {l['text']}" for l in similar_logs[:3]])
    prompt = (
        f"Anomalous network log detected:\n\n'{log_text}'\n\n"
        f"Most similar known patterns:\n{context}\n\n"
        "In 3 sentences: What might this log indicate? What should the NOC engineer check?"
    )
    msg = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=200,
        system="You are a network security analyst. Be concise and actionable.",
        messages=[{"role": "user", "content": prompt}]
    )
    return msg.content[0].text

class SmartLogEngine:
    """
    Full log analysis engine:
    - Hybrid search (semantic + BM25)
    - Anomaly detection
    - LLM explanation for anomalies
    """

    def __init__(self):
        texts = [l["text"] for l in NETWORK_LOGS]
        _, self._tfidf_vecs = build_tfidf_index(texts)

        self._bm25 = BM25()
        self._bm25.fit(texts)

        self._detector = VectorAnomalyDetector(k_clusters=4, anomaly_percentile=90)
        self._detector.fit(NETWORK_LOGS)

    def search(self, query: str, top_k: int = 3) -> list:
        """Hybrid search: semantic + BM25 merged with RRF."""
        _, [qvec] = build_tfidf_index([query])
        sem = sorted([(cosine_similarity(qvec, v), i)
                      for i, v in enumerate(self._tfidf_vecs)],
                     reverse=True)
        sem_top = [(s, i) for s, i in sem[:10] if s > 0.01]
        bm25    = self._bm25.search(query, top_k=10)
        hybrid  = reciprocal_rank_fusion(sem_top, bm25, NETWORK_LOGS, top_k=top_k)
        return [{"rrf_score": round(s, 4), "log": l} for s, l in hybrid]

    def ingest_and_check(self, new_log: str) -> dict:
        """Check if a new incoming log is anomalous."""
        anom  = self._detector.detect(new_log)
        # Find similar known logs for context
        similar_results = self.search(new_log, top_k=3)
        similar_logs    = [r["log"] for r in similar_results]
        anom["similar_known_logs"] = similar_logs
        if anom["is_anomaly"]:
            anom["llm_analysis"] = llm_analyze(new_log, similar_logs)
        return anom

engine = SmartLogEngine()

print("SMART LOG ENGINE — Full Integration Test")
print("=" * 65)

# ── Test 1: Hybrid search ─────────────────────────────────────────────────────
print("\n[HYBRID SEARCH]")
print("Query: 'BGP peer lost connection 10.0.0.2'")
results = engine.search("BGP peer lost connection 10.0.0.2", top_k=3)
for r in results:
    log = r["log"]
    print(f"  [{r['rrf_score']:.4f}] [{log['category']}] {log['text'][:60]}")

# ── Test 2: Normal logs ───────────────────────────────────────────────────────
print("\n[ANOMALY DETECTION — Normal logs]")
normal_test = [
    "BGP session reset with AS 65003",
    "SSH failed login from 10.0.1.1",
]
for log_text in normal_test:
    r = engine.ingest_and_check(log_text)
    print(f"  {'🚨 ANOMALY' if r['is_anomaly'] else '✅ normal ':10s} (score={r['anomaly_score']:.1f}x) {log_text[:55]}")

# ── Test 3: Anomalous logs ────────────────────────────────────────────────────
print("\n[ANOMALY DETECTION — Novel logs]")
anomaly_tests = [
    "BGP UPDATE with malformed AS_PATH attribute type 255",
    "Kernel panic: memory corruption in IP forwarding process",
]
for log_text in anomaly_tests:
    print(f"\n  Log: '{log_text}'")
    r = engine.ingest_and_check(log_text)
    print(f"  {'🚨 ANOMALY' if r['is_anomaly'] else '✅ normal '} (anomaly_score={r['anomaly_score']:.1f}x threshold)")
    if r["is_anomaly"] and "llm_analysis" in r:
        print(f"  LLM Analysis:")
        for line in r["llm_analysis"].split("\n")[:4]:
            print(f"    {line}")

print()
print("=" * 65)
print("ENGINE CAPABILITIES COMBINED:")
print("  1. TF-IDF vectors  → semantic representation (no GPU needed)")
print("  2. IVF clustering  → 10-100x faster than brute-force search")
print("  3. HNSW concept    → best recall at scale (use FAISS in production)")
print("  4. Hybrid RRF      → exact IPs + semantic meaning in one query")
print("  5. Anomaly detect  → catches novel threats that rules miss")
print("  6. LLM explanation → actionable insights for NOC engineers")
print()
print("✅ Chapter 35 complete — Vector Database Optimization mastered!")
